In [ ]:
using Pkg
using CSV
using DataFrames
using Plots
using Printf

In [ ]:
function read_mfem_nodes(filename)
    coords = Float64[]
    in_nodes = false
    skip = 0

    for line in eachline(filename)
        if strip(line) == "nodes"
            in_nodes = true
            skip = 4  # skip FE space metadata
            continue
        end

        if in_nodes
            if skip > 0
                skip -= 1
                continue
            end
            try
                push!(coords, parse(Float64, strip(line)))
            catch
            end
        end
    end
    return coords
end

In [ ]:
function read_other(filename)
    coords = Float64[]
    in_nodes = false
    skip = 4

    for line in eachline(filename)
        if skip > 0
            skip -= 1
            continue
        end
        try
            push!(coords, parse(Float64, strip(line)))
        catch
        end
    end
    return coords
end

In [ ]:
function read_all(prob, rs, ode, tf, igr = true, alpha = 4e-6, suffix = "")
    folder = (igr ? "WithIGR/" * "alpha=" * (@sprintf "%1.6f" alpha*1e6) * "e-6/" : "WithoutIGR/") * "p" * string(prob) * "/"
    str = string(prob) * "_" * string(rs) * "_" * string(ode) * "_" * string(tf) 
    if(!igr)
        str = str * "_noigr"
    end
    str = str * suffix
    x = read_mfem_nodes(folder * "Laghos_" * str * "_mesh")
    rho = read_other(folder * "Laghos_" * str * "_rho")
    v = read_other(folder * "Laghos_" * str * "_v")
    e = read_other(folder * "Laghos_" * str * "_e")
    igrp = read_other(folder * "Laghos_" * str * "_igr")

    pv = sortperm(x)
    x = x[pv]; rho = rho[pv]; v = v[pv]; e = e[pv]
    p = 0.4 .* rho .* e

    return x, rho, v, e, p, igrp
end

In [ ]:
function plot_igr(vals, axis, steps, increment, prob, rs, ode, tf, igr = true, xlim = (0,1), ylim = (0,1), sep = 0, alpha = 4e-6, suffix = "")

    choose_y(vals) = (vals == "rho" ? rho : (vals == "v" ? v : (vals == "e" ? e : (vals == "p" ? p : igrp))))

    out = plot()
    if(axis == "tf")
        x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, igr, alpha, suffix)
        plot!(x, choose_y(vals), xlim = xlim, ylim = ylim, lab = "tf = " * string(tf), title = vals * " " * suffix)
    elseif(axis == "rs")
        x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, igr, alpha, suffix)
        plot!(x, choose_y(vals), xlim = xlim, ylim = ylim, lab = "RS " * string(rs), title = vals * " at t = " * string(tf) * " " * suffix)
    elseif(axis == "igr")
        x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, true, alpha, suffix)
        plot!(x, choose_y(vals), xlim = xlim, ylim = ylim, lab = "IGR", title = vals * " at t = " * string(tf) * " " * suffix)
    elseif(axis == "alpha")
        x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, true, alpha, suffix)
        plot!(x, choose_y(vals), xlim = xlim, ylim = ylim, lab = alpha, title = vals * " at t = " * string(tf) * " " * suffix)
    elseif(axis == "RSalpha")
        x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, true, alpha, suffix)
        plot!(x, choose_y(vals), xlim = xlim, ylim = ylim, lab = "RS " * string(rs) * ", alpha= " * string(alpha), title = vals * " at t = " * string(tf) * " " * suffix)
    end 
    plot!(ylab=vals)
    
    for i in 2:steps
        if(axis == "tf")
            x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf + increment*(i-1), igr, alpha, suffix)
            plot!(x, choose_y(vals) .+ sep*(i-1), xlim = xlim, ylim = ylim, lab = "tf = " * string(tf + increment*(i-1)))
        elseif(axis == "rs")
            x, rho, v, e, p, igrp = read_all(prob, rs + (i-1), ode, tf, igr, alpha, suffix)
            plot!(x, choose_y(vals) .+ sep*(i-1), xlim = xlim, ylim = ylim, lab = "RS " * string(rs + (i-1)))
        elseif(axis == "igr")
            x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, false, alpha, suffix)
            plot!(x, choose_y(vals) .+ sep*(i-1), xlim = xlim, ylim = ylim, lab = "NO IGR")
            break
        elseif(axis == "alpha")
            x, rho, v, e, p, igrp = read_all(prob, rs, ode, tf, true, alpha / 4^(i-1), suffix)
            plot!(x, choose_y(vals) .+ sep*(i-1), xlim = xlim, ylim = ylim, 
                                lab = "RS " * string(rs) * ", alpha= " * string(alpha/4^(i-1)))
        elseif(axis == "RSalpha")
            x, rho, v, e, p, igrp = read_all(prob, rs + (i-1), ode, tf, true, alpha / 4^(i-1), suffix)
            plot!(x, choose_y(vals) .+ sep*(i-1), xlim = xlim, ylim = ylim, 
                                lab = "RS " * string(rs + (i-1)) * ", alpha= " * string(alpha/4^(i-1)))
        end 
    end
    
    return out
end

In [ ]:
#           vals, axis, steps, increment, prob, rs, ode, tf, igr, xlim, ylim, sep = 0, alpha = "alp...", suffix = ""
plot1 = plot_igr("rho", "RSalpha", 1, 1, 14, 9, 4, 0.2, true, (-1.0,2.0), (-0.2,5.3), 0.3, 16e-6, "_100.000000")
plot!(title = "Mach Test, Visc Const 100, t = 0.2")

In [ ]:
plot1 = plot_igr("v", "tf", 0.01, 1, 0.01, 14, 11, 4, 0.05, true, (0.3,0.45), (1.5,3.4), 0, "_0.001000")
plot!(title = "Visc 1e-3")
plot2 = plot_igr("v", "tf", 0.01, 1, 0.01, 14, 11, 4, 0.05, true, (0.3,0.45), (1.5,3.4), 0, "_0.000100")
plot!(title = "Visc 1e-4")

In [ ]:
plot!(plot1, plot2, ylab = "Velocity")

In [ ]:
x, rho, v, e, p, igrp = read_all(14, 12, 4, 0.02, true, "_LaghosVisc");
plot(x, rho .+ .1, xlim=(0.3,0.5), lab = "LaghosViscIGR")
x, rho, v, e, p, igrp = read_all(14, 12, 4, 0.02, false, "_LaghosVisc");
plot!(x, rho .+ .2, xlim=(0.3,0.5), lab = "LaghosVisc NO IGR")

In [ ]:
x, rho, v, e, p, igrp = read_all(14, 11, 4, 0.05, true, "_0.001000");
plot(x, rho, xlim=(0.3,0.5), title = "Density by Viscosity", lab = "Visc 1e-3", ylab = "Density")
x, rho, v, e, p, igrp = read_all(14, 11, 4, 0.05, true, "_LaghosVisc");
plot!(x, rho, xlim=(0.3,0.5), title = "Density by Viscosity", lab = "LaghosVisc", ylab = "Density")

In [ ]:
x, rho, v, e, p, igrp = read_all(14, 12, 4, 0.05, true, "_0.000330");
plot(x, rho, xlim=(0.3,0.5), title = "Density by Viscosity", lab = "Visc 1e-4", ylab = "Density")

In [ ]:
x, rho, v, e, p, igrp = read_all(14, 11, 4, 0.05, true, "_0.000100");
plot(x, rho, xlim=(0.3,0.5), title = "Density by Viscosity", lab = "Visc 1e-4", ylab = "Density")
x, rho, v, e, p, igrp = read_all(14, 11, 4, 0.05, true, "_0.001000");
plot!(x, rho, lab = "Visc 1e-3")

In [ ]:
# Read Brook
data = CSV.read("snapshot_smooth_shu_osher_p1/snapshot_smooth_shu_osher_p1/t_0-1/num_smooth_shu_osher_rk3_1d_m_4096.csv", DataFrame)  # if using CSV.jl
xB = data[:,1]; rhoB = data[:,2]; muB = data[:,3]; EB = data[:,4];
vB = data[:,5]; pB = data[:,6]; SigmaB = data[:,7]; eB = data[:,8];

In [ ]:
# Can also compare to exact Riemann solver sometimes from Julia
include("RiemannSolver.jl")

In [ ]:
#                   rho,   u,  p, gamma
left  = HydroStatus(1.1, 0.0, 0.11, 1.4)
right = HydroStatus(0.1, 0.0, 0.01, 1.4)
xs = range(0.0, 1.0, length=2052)
t  = 0.2

states = sample_riemann(xs .- 0.5, t, left, right);

In [ ]:
ExRho = getfield.(states, :rho)
ExU = getfield.(states, :u)
ExP = getfield.(states, :p)
ExE = ExP ./ (0.4 .* ExRho);

In [ ]:
p1 = plot(xs, ExRho, title = "Density")
p2 = plot(xs, ExU, title = "Velocity")
p3 = plot(xs, ExE, title = "Internal Energy")
p4 = plot(xs, ExP, title = "Pressure")

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
data = CSV.read("snapshot_smooth_shu_osher_p1/snapshot_smooth_shu_osher_p1/t_0-2/num_smooth_shu_osher_rk3_1d_m_4096.csv", DataFrame)  # if using CSV.jl
xB = data[:,1]; rhoB = data[:,2]; muB = data[:,3]; EB = data[:,4];
vB = data[:,5]; pB = data[:,6]; SigmaB = data[:,7]; eB = data[:,8];

In [ ]:
#plot(xB, yB, rhoB, st =:surface, camera=(0, 90))

In [ ]:
p1 = plot(x, rho, title = "Density", label = "")
plot!(xB, rhoB)
p2 = plot(x, v, title = "Velocity", label = "")
plot!(xB, vB)
p3 = plot(x, e, title = "Internal Energy", label = "")
plot!(xB, eB)
p4 = plot(x, p, title = "Pressure", label = "")
plot!(xB, pB)

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
#Interpolate Functions

ConvInt(a, b, c) = (b-a)/(c-a)
ConvAvg(a, b, L) = (1-L)*a + L*b 

point = 1
stay = true
igrp2 = []
for i in 1:length(xB)-1
    while(stay)
        if(x[point + 1] > xB[i])
            L = ConvInt(x[point], xB[i], x[point + 1])
            push!(igrp2, ConvAvg(igrp[point], igrp[point + 1], L))
            stay = false
        else
            point += 1
        end
    end
    stay = true
end